<a href="https://colab.research.google.com/github/Nova3012k/Deep-learning-2026/blob/main/Hands_On_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Clasificación de Flores Iris con Multilayer Perceptron (MLP)**

1. Descripción del problema de clasificación por enfrentar

El objetivo es clasificar plantas de Iris en una de tres especies posibles (Setosa, Versicolor o Virginica) basándose en cuatro características físicas de sus flores. Es un problema de clasificación multiclase clásico en el que buscamos que la red neuronal aprenda los límites de decisión entre las especies a partir de datos históricos.

# **2. Pipeline**


## **2.1 Exploración y preprocesamiento del Dataset**

## **2.1.1 Descripción de las variables:tipos e interpretación**
Variables Numéricas (Features): sepal-length, sepal-width, petal-length, petal-width. Representan medidas físicas en centímetros.

Variable Categórica (Target): Class. Representa el nombre de la especie.

In [1]:
import pandas as pd
from sklearn import preprocessing

# Carga de datos desde la URL oficial
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
names = ['sepal-length', 'sepal-width', 'petal-length', 'petal-width', 'Class']
irisdata = pd.read_csv(url, names=names)

print(irisdata.head())

   sepal-length  sepal-width  petal-length  petal-width        Class
0           5.1          3.5           1.4          0.2  Iris-setosa
1           4.9          3.0           1.4          0.2  Iris-setosa
2           4.7          3.2           1.3          0.2  Iris-setosa
3           4.6          3.1           1.5          0.2  Iris-setosa
4           5.0          3.6           1.4          0.2  Iris-setosa


## **2.1.2 Variables independientes y dependiente**
Variables Independientes (X): Las primeras 4 columnas con las medidas físicas.

Variable Dependiente (y): La quinta columna (Class). Como es texto, aplicamos LabelEncoder para convertirla a números (0, 1, 2).

In [2]:
X = irisdata.iloc[:, 0:4]
y = irisdata.select_dtypes(include=[object])

# Transformación de categorías a valores numéricos
le = preprocessing.LabelEncoder()
y = y.apply(le.fit_transform)

print("Clases únicas detectadas:", irisdata.Class.unique())

Clases únicas detectadas: ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


## **2.1.3 Estandarización: ¿Por qué y cómo?**

¿Por qué?: Las redes neuronales utilizan el descenso del gradiente. Si las variables tienen rangos muy distintos, el algoritmo puede "perderse" o tardar demasiado en converger. La estandarización asegura que todas las características contribuyan por igual al aprendizaje.

¿Cómo?: Se utiliza la fórmula de puntuación Z:
$$z = \frac{x - \mu}{\sigma}$$Esto centra los datos en una media de 0 con una desviación estándar de 1.

In [3]:
from sklearn.preprocessing import StandardScaler

# Ejemplo rápido de cómo cambia un dato
scaler_example = StandardScaler()
data_test = [[5.1], [7.0], [4.5]]
print("Ejemplo antes y después:\n", scaler_example.fit_transform(data_test))

Ejemplo antes y después:
 [[-0.40664731]
 [ 1.37634474]
 [-0.96969743]]


## **2.2 Model Selection (Arquitectura de la MLP)**
Para este problema, el artículo propone una arquitectura de 3 capas ocultas, cada una con 10 neuronas (10, 10, 10).

Esta profundidad permite capturar las diferencias sutiles entre las clases Versicolor y Virginica, que suelen estar muy solapadas.

Se utiliza un máximo de 1000 iteraciones para garantizar que la red tenga tiempo suficiente de minimizar el error.

## **2.3 Model Training**
Aquí realizamos la división de datos (80% entrenamiento / 20% prueba), escalamos y entrenamos.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# 1. División Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)

# 2. Escalado de características
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# 3. Creación y entrenamiento del modelo
mlp = MLPClassifier(hidden_layer_sizes=(10, 10, 10), max_iter=1000)
mlp.fit(X_train, y_train.values.ravel())

print("¡Modelo entrenado exitosamente!")

¡Modelo entrenado exitosamente!


## **2.4 Model Prediction**

Construimos la función solicitada para verificar que la red clasifica correctamente un patrón de entrada.

In [7]:
def prediccion(modelo, scaler_obj, nuevo_patron, nombres_columnas):

    import pandas as pd
    import numpy as np

    # 1. Convertimos la lista en un DataFrame con los nombres originales
    patron_df = pd.DataFrame([nuevo_patron], columns=nombres_columnas)

    patron_scaled = scaler_obj.transform(patron_df)

    # 3. Predecimos
    pred = modelo.predict(patron_scaled)

    # 4. Convertimos el número a nombre de flor
    nombre_clase = le.inverse_transform(pred)
    return nombre_clase[0]

# X.columns contiene ['sepal-length', 'sepal-width', 'petal-length', 'petal-width']
ejemplo_crudo = [5.1, 3.5, 1.4, 0.2]
resultado = prediccion(mlp, scaler, ejemplo_crudo, X.columns)

print(f"Patrón de entrada: {ejemplo_crudo} -> Predicción de la Red: {resultado}")

Patrón de entrada: [5.1, 3.5, 1.4, 0.2] -> Predicción de la Red: Iris-setosa


## **2.5 Model Evaluation**


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = mlp.predict(X_test)

print("MATRIZ DE CONFUSIÓN:")
print(confusion_matrix(y_test, predictions))

print("\nREPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, predictions))

MATRIZ DE CONFUSIÓN:
[[13  0  0]
 [ 0  8  0]
 [ 0  1  8]]

REPORTE DE CLASIFICACIÓN:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       0.89      1.00      0.94         8
           2       1.00      0.89      0.94         9

    accuracy                           0.97        30
   macro avg       0.96      0.96      0.96        30
weighted avg       0.97      0.97      0.97        30

